# Block A: Data Discovery & Validation

Objective: Understand the exact structure, availability, and quality of the CTU-CHB dataset before any processing.

This notebook explores:
- Signal inventory (FHR, UC, sampling rates, durations)
- Outcome labels availability (pH, Apgar scores, neonatal outcomes, normal/pathological classification)
- Data quality metrics (dropouts, signal quality, sampling consistency)
- Class distribution

In [25]:
import os
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Try to import wfdb, install if not available
try:
    import wfdb
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'wfdb'])
    import wfdb

print("Libraries imported successfully")

sys.executable: /home/naem_haq/miniconda3/envs/ml/bin/python
python version: 3.10.19
cwd: /home/naem_haq/projects/CTG/notebooks
home: /home/naem_haq
platform: Linux 6.6.87.2-microsoft-standard-WSL2
listing project data dir: False no data folder here


In [26]:
def parse_hea_file(hea_path):
    """
    Parse .hea header file to extract metadata.
    Returns signal information, recording details, and outcomes.
    """
    metadata = {
        'record_name': None,
        'num_signals': 0,
        'sampling_rate': None,
        'num_samples': 0,
        'duration_seconds': 0,
        'signals': [],
        'fhr_available': False,
        'uc_available': False,
        'mhr_available': False,
        'fhr_sampling_rate': None,
        'uc_sampling_rate': None,
        'outcomes': {}
    }
    
    try:
        with open(hea_path, 'r') as f:
            lines = f.readlines()
        
        # First line: record name, num signals, sampling rate, num samples
        first_line = lines[0].strip().split()
        metadata['record_name'] = first_line[0]
        metadata['num_signals'] = int(first_line[1])
        metadata['sampling_rate'] = int(first_line[2])
        metadata['num_samples'] = int(first_line[3])
        
        # Calculate duration in seconds
        if metadata['sampling_rate'] > 0:
            metadata['duration_seconds'] = metadata['num_samples'] / metadata['sampling_rate']
        
        # Parse signal lines (one per signal)
        for i in range(metadata['num_signals']):
            if i + 1 < len(lines):
                signal_line = lines[i + 1].strip()
                # Format: file format gain baseline sample_units ADC_resolution ADC_zero initial_value checksum
                parts = signal_line.split()
                if len(parts) >= 1:
                    signal_info = {
                        'name': parts[0] if len(parts) > 0 else f'Signal_{i}',
                        'file_format': parts[0].split('.')[1] if '.' in parts[0] else 'unknown',
                        'description': ' '.join(parts[1:]) if len(parts) > 1 else ''
                    }
                    metadata['signals'].append(signal_info)
                    
                    # Check signal type
                    signal_name = signal_info['name'].lower()
                    if 'fhr' in signal_name or signal_name == 'fhr.dat':
                        metadata['fhr_available'] = True
                        metadata['fhr_sampling_rate'] = metadata['sampling_rate']
                    elif 'uc' in signal_name or signal_name == 'uc.dat':
                        metadata['uc_available'] = True
                        metadata['uc_sampling_rate'] = metadata['sampling_rate']
                    elif 'mhr' in signal_name or 'maternal' in signal_name.lower():
                        metadata['mhr_available'] = True
        
        # Parse comments for outcomes (typically start with #)
        for line in lines[metadata['num_signals'] + 1:]:
            line = line.strip()
            if line.startswith('#'):
                # Parse outcome information
                line_clean = line.lstrip('#').strip()
                if 'Outcome' in line_clean or 'outcome' in line_clean:
                    parts = line_clean.split(':')
                    if len(parts) == 2:
                        key = parts[0].strip()
                        value = parts[1].strip()
                        metadata['outcomes'][key] = value
                # Common fields in CTU-CHB dataset
                elif any(field in line_clean for field in ['pH', 'Apgar', 'SBS', 'REC TYPE']):
                    parts = line_clean.split(':')
                    if len(parts) == 2:
                        key = parts[0].strip()
                        value = parts[1].strip()
                        metadata['outcomes'][key] = value
    
    except Exception as e:
        print(f"Error parsing {hea_path}: {str(e)}")
    
    return metadata

# Test on first the first file to verify parsing works correctly
test_file = os.path.join(data_dir, sorted(hea_files)[0]) 
test_metadata = parse_hea_file(test_file)
print(f"\n✓ Successfully parsed: {test_file}")
print(f"  Recording: {test_metadata['record_name']}")
print(f"  Signals: {test_metadata['num_signals']}")
print(f"  Duration: {test_metadata['duration_seconds']:.1f} seconds ({test_metadata['duration_seconds']/60:.1f} minutes)")
print(f"  FHR available: {test_metadata['fhr_available']}")
print(f"  UC available: {test_metadata['uc_available']}")
print(f"  Outcomes: {test_metadata['outcomes']}")

Libraries imported successfully


In [27]:
def extract_outcome_label(outcomes_dict):
    """
    Extract the primary outcome label (normal vs pathological) from outcomes dictionary.
    Common fields: 'REC TYPE', 'Outcome', 'outcome'
    """
    label = None
    label_field = None
    
    for key, value in outcomes_dict.items():
        value_lower = str(value).lower().strip()
        # Look for REC TYPE field (typically contains N or P)
        if 'rec type' in key.lower():
            label_field = key
            if 'n' in value_lower or 'normal' in value_lower:
                label = 'Normal'
            elif 'p' in value_lower or 'path' in value_lower:
                label = 'Pathological'
        # Look for explicit outcome field
        elif 'outcome' in key.lower():
            label_field = key
            if 'normal' in value_lower:
                label = 'Normal'
            elif 'path' in value_lower or 'abnormal' in value_lower:
                label = 'Pathological'
    
    return label, label_field

# Test extraction
print("Testing outcome extraction...")
test_label, test_field = extract_outcome_label(test_metadata['outcomes'])
print(f"  Outcome label: {test_label}")
print(f"  Field: {test_field}")
print(f"  Raw outcomes: {test_metadata['outcomes']}")

✓ Data directory found: /home/naem_haq/projects/CTG/data/ctu-chb-intrapartum-cardiotocography-database-1.0.0
✓ Total .hea files: 552
✓ Sample files: ['1001.hea', '1002.hea', '1003.hea', '1004.hea', '1005.hea']


In [28]:
# Create DataFrame for easier analysis
df = pd.DataFrame(all_records)

print("=" * 80)
print("SIGNAL INVENTORY")
print("=" * 80)
print(f"\nSignal Availability:")
for signal_type, count in sorted(signal_inventory.items()):
    pct = 100 * count / signal_inventory['Total']
    print(f"  {signal_type:20s}: {count:4d} / {signal_inventory['Total']} ({pct:5.1f}%)")

print("\n" + "=" * 80)
print("RECORDING DURATION STATISTICS")
print("=" * 80)
print(f"\nDuration (minutes):")
print(f"  Mean:     {df['duration_minutes'].mean():.2f}")
print(f"  Median:   {df['duration_minutes'].median():.2f}")
print(f"  Std Dev:  {df['duration_minutes'].std():.2f}")
print(f"  Min:      {df['duration_minutes'].min():.2f}")
print(f"  Max:      {df['duration_minutes'].max():.2f}")
print(f"  Total:    {df['duration_minutes'].sum():.0f} minutes ({df['duration_minutes'].sum()/60:.1f} hours)")

print("\n" + "=" * 80)
print("OUTCOME LABEL DISTRIBUTION")
print("=" * 80)
outcome_counts = df['outcome_label'].value_counts()
print(f"\nClass Distribution:")
for outcome, count in outcome_counts.items():
    if pd.notna(outcome):
        pct = 100 * count / len(df)
        print(f"  {outcome:20s}: {count:4d} ({pct:5.1f}%)")

unknown_count = df['outcome_label'].isna().sum()
if unknown_count > 0:
    pct = 100 * unknown_count / len(df)
    print(f"  {'Unknown/Missing':20s}: {unknown_count:4d} ({pct:5.1f}%)")


✓ Successfully parsed: /home/naem_haq/projects/CTG/data/ctu-chb-intrapartum-cardiotocography-database-1.0.0/1001.hea
  Recording: 1001
  Signals: 2
  Duration: 4800.0 seconds (80.0 minutes)
  FHR available: False
  UC available: False
  Outcomes: {}


In [29]:
print("\n" + "=" * 80)
print("SUMMARY STATISTICS TABLE")
print("=" * 80)

summary_stats = {
    'Metric': [
        'Total recordings analyzed',
        'Mean recording duration (minutes)',
        'Median recording duration (minutes)',
        'Recordings with FHR signal',
        'Recordings with UC signal',
        'Recordings with MHR signal',
        'Normal cases',
        'Pathological cases',
        'Unknown/unlabeled cases',
        'Records passing quality criteria (≥50% valid FHR)',
        'Records failing quality criteria (<50% valid FHR)',
        'Mean FHR missing data (%)',
        'Mean FHR outliers (%)',
        'Mean UC missing data (%)',
        'Mean UC outliers (%)'
    ],
    'Value': [
        len(df),
        f"{df['duration_minutes'].mean():.2f}",
        f"{df['duration_minutes'].median():.2f}",
        f"{df['fhr_available'].sum()}",
        f"{df['uc_available'].sum()}",
        f"{df['mhr_available'].sum()}",
        f"{(df['outcome_label'] == 'Normal').sum()}",
        f"{(df['outcome_label'] == 'Pathological').sum()}",
        f"{df['outcome_label'].isna().sum()}",
        f"{quality_pass}",
        f"{quality_fail}",
        f"{df['fhr_missing_pct'].mean():.2f}%",
        f"{df['fhr_outliers_pct'].mean():.2f}%",
        f"{df['uc_missing_pct'].mean():.2f}%",
        f"{df['uc_outliers_pct'].mean():.2f}%"
    ]
}

summary_df = pd.DataFrame(summary_stats)
print("\n" + summary_df.to_string(index=False))

print("\n" + "=" * 80)

Testing signal loading and quality assessment...
  FHR Loaded: True, Length: 19200
  UC Loaded: True, Length: 19200
  FHR Missing: 0.00%
  UC Missing: 0.00%
